In [1]:
from tqdm.notebook import tqdm
import polars as pl

# wags-llm imports
from wags_llm.cache import InMemoryCache
from wags_llm.client import BedrockClaudeJsonClient
from wags_llm.prompts import build_empty_registry
from wags_llm.services import StructuredTaskRunner

# dgiLIT Pydantic Models
from dgilit_models import InteractionClassificationPrompt
from dgilit_models import InteractionClassificationResult


# wags-llm
Wags-LLM exists to add structure to LLM prompt calls in a repeatable, explainable way. This is the first attempt at implementing this with a dgiLIT prompt. Other portions of the dgiLIT pipeline would happen before this step and as such, the supporting models and prompts in this branch will likely evolve as those are tuned up.  
  
Notable things to update:  
- pre-tagging of drugs and genes to submit alongside a prompt. 
- support for multiple interactions per one abstract or block of text
- support for chunking sections of a PMID (Abstract, Results, Conclusion)
- support for Pydantic model level sanity checks to reduce LLM usage where not needed (may involve pre-tagging or deterministic rule sets that would provide a `run_llm? TRUE FALSE` type evaluation)

In [2]:
MODEL_ID = "us.anthropic.claude-sonnet-4-6"
REGION_NAME = "us-east-1"
PROFILE_NAME = "dev-account"
MAX_TOKENS = 1000

def build_llm_task_runner(
    model_id: str,
    region_name: str,
    profile_name: str,
    max_tokens: int,
    temperature: float,
) -> StructuredTaskRunner:
    """Build LLM interaction extraction task runner

    :param model_id: Bedrock model identifier.
    :param region_name: AWS region for the Bedrock runtime client.
    :param profile_name: AWS profile name.
    :param max_tokens: Maximum number of tokens to request from the model.
    :param temperature: Sampling temperature.
    :return: Configured structured task runner instance.
    """
    registry = build_empty_registry()
    registry.register(InteractionClassificationPrompt(version="v2"))
    llm_client = BedrockClaudeJsonClient(
        model_id=model_id,
        region_name=region_name,
        profile_name=profile_name,
        max_tokens=max_tokens,
        temperature=temperature,
    )
    cache = InMemoryCache()
    return StructuredTaskRunner(
        client=llm_client, prompt_registry=registry, cache=cache
    )

In [3]:
def classify_interaction(
    task_runner: StructuredTaskRunner,
    prompt: InteractionClassificationPrompt,
    context: str,
    candidate_drug: str,
    candidate_gene: str,
) -> InteractionClassificationResult:
    """Classify whether one candidate drug-gene pair is supported by biomedical text."""

    payload = prompt.build_payload(
        context=context,
        candidate_drug=candidate_drug,
        candidate_gene=candidate_gene,
    )

    try:
        task_result = task_runner.execute(
            prompt_name=prompt.name,
            prompt_version=prompt.version,
            payload=payload,
            response_model=InteractionClassificationResult,
        )

        return task_result

    except Exception as e:
        return InteractionClassificationResult(
            drug=candidate_drug,
            gene=candidate_gene,
            interaction=False,
            evidence=None,
            interaction_type=None,
            directionality=None,
            error_message=str(e),
        )

### Pre-Tagging


Description

In [4]:
df = pl.read_excel("../2026-06-08-test1-EGFR-only.xlsx") # file with pmids + context blocks

In [5]:
from dgilit_wags.tagging import (
    BioBertEntityTagger,
    EntityPreTaggingService,
    SQLiteNormalizerCache,
    TaggerConfig,
    ViccNormalizer,
)

pretagger = EntityPreTaggingService(
    tagger=BioBertEntityTagger(
        TaggerConfig(
            batch_size=16,
            include_drugs=True,
            include_genes=True,
            include_diseases=False,
            device=-1,  # use 0 on CUDA
        )
    ),
    normalizer=ViccNormalizer(
        cache=SQLiteNormalizerCache(".dgilit_normalizer_cache.sqlite")
    ),
)

In [6]:
contexts = (
    df["context"]
    .fill_null("")
    .cast(pl.Utf8)
    .to_list()
)
pmids = df["pmid"].cast(str).to_list() if "pmid" in df.columns else [None] * len(contexts)
block_ids = [str(i) for i in range(len(contexts))]

tagged_blocks = pretagger.tag_blocks(
    contexts=contexts,
    pmids=pmids,
    block_ids=block_ids,
)

Device set to use cpu


Tagging text batches:   0%|          | 0/3 [00:00<?, ?it/s]

Device set to use cpu


Tagging text batches:   0%|          | 0/3 [00:00<?, ?it/s]

Normalizing unique entities:   0%|          | 0/149 [00:00<?, ?entity/s]

In [7]:
def entity_display_name(e):
    if e.concept and e.concept.concept_label:
        return e.concept.concept_label
    if e.text:
        return e.text
    return None


def build_candidate_context(block):
    drugs = sorted({
        e.concept.concept_label
        for e in block.entities
        if e.entity_type == "drug"
        and e.concept
        and e.concept.concept_label
        and e.concept.concept_id
    })

    genes = sorted({
        e.concept.concept_label
        for e in block.entities
        if e.entity_type == "gene"
        and e.concept
        and e.concept.concept_label
        and e.concept.concept_id
    })

    return {
        "pmid": block.pmid,
        "drugs": drugs,
        "genes": genes,
        "context": block.context,
    }

candidate = build_candidate_context(tagged_blocks[0])
candidate

{'pmid': '19969465',
 'drugs': ['activated charcoal', 'tyrosine'],
 'genes': ['EGF', 'EGFR'],
 'context': 'A series of 4-anilinoquinazolines with C-C multiple bond substitutions at the 6-position were synthesized and investigated for their potential to inhibit epidermal growth factor receptor (EGFR) tyrosine kinase activity. Among the compounds synthesized, alkyne 6d and allenes 7d and 7f significantly inhibited EGFR tyrosine kinase activity. These compounds inhibited EGF-mediated phosphorylation of EGFR in A431 cells, resulting in cell-cycle arrest and apoptosis induction. The C-C multiple bonds substituted at the C-6 position of the anilinoquinazoline framework were essential for the significant inhibitory activity. Compounds with long carbon chains (n=3-6), such as 6c-f, 7c-f, 11, and 12, displayed prolonged inhibitory activity.'}

### LLM Calls


Description

In [8]:
def entity_display_name(e):
    if e.concept and e.concept.concept_label and e.concept.concept_id:
        return e.concept.concept_label
    return None


def get_candidates(block):
    drugs = sorted({
        name
        for e in block.entities
        if e.entity_type == "drug"
        for name in [entity_display_name(e)]
        if name
    })

    genes = sorted({
        name
        for e in block.entities
        if e.entity_type == "gene"
        for name in [entity_display_name(e)]
        if name
    })

    return drugs, genes

In [9]:
from itertools import product

temperatures = [0]

def run_experiments(tagged_blocks, temperatures, num_runs, prompt_version: str):
    stored_runs = []

    for temp in temperatures:
        for run_idx in range(num_runs):
            task_runner = build_llm_task_runner(
                MODEL_ID,
                REGION_NAME,
                PROFILE_NAME,
                MAX_TOKENS,
                temp,
            )

            prompt = InteractionClassificationPrompt(version=prompt_version)

            print(f"Running temp={temp}, run={run_idx + 1}")

            results = []

            for block in tqdm(
                tagged_blocks,
                total=len(tagged_blocks),
                desc=f"T={temp}, run={run_idx + 1}",
                leave=False,
            ):
                candidate_drugs, candidate_genes = get_candidates(block)

                if not candidate_drugs or not candidate_genes:
                    results.append(
                        {
                            "pmid": block.pmid,
                            "block_id": block.block_id,
                            "skipped": True,
                            "skip_reason": "No normalized drug and gene candidates",
                            "candidate_drugs": candidate_drugs,
                            "candidate_genes": candidate_genes,
                            "classifications": [],
                        }
                    )
                    continue

                classifications = []

                for candidate_drug, candidate_gene in product(
                    candidate_drugs,
                    candidate_genes,
                ):
                    result = classify_interaction(
                        task_runner=task_runner,
                        prompt=prompt,
                        context=block.context,
                        candidate_drug=candidate_drug,
                        candidate_gene="EGFR", # candidate_gene when bulk extraction, seed gene otherwise
                    )

                    classifications.append(result.model_dump())

                results.append(
                    {
                        "pmid": block.pmid,
                        "block_id": block.block_id,
                        "skipped": False,
                        "candidate_drugs": candidate_drugs,
                        "candidate_genes": candidate_genes,
                        "num_candidate_pairs": len(candidate_drugs) * len(candidate_genes),
                        "classifications": classifications,
                    }
                )

            stored_runs.append(
                {
                    "run_idx": run_idx,
                    "prompt_version": prompt_version,
                    "temperature": temp,
                    "results": results,
                }
            )

            print(f"Done temp={temp}, run={run_idx + 1}")

    return stored_runs

In [10]:
stored_runs = run_experiments(
    tagged_blocks,
    temperatures,
    num_runs=1,
    prompt_version="v2",
)

Running temp=0, run=1


T=0, run=1:   0%|          | 0/38 [00:00<?, ?it/s]

Done temp=0, run=1


NOTE: After running this once, temperature as a parameter really doesn't need to be included. Kinda just set it to 0 and forget about it tbh.

In [12]:
stored_runs



[{'run_idx': 0,
  'prompt_version': 'v2',
  'temperature': 0,
  'results': [{'pmid': '19969465',
    'block_id': '0',
    'skipped': False,
    'candidate_drugs': ['activated charcoal', 'tyrosine'],
    'candidate_genes': ['EGF', 'EGFR'],
    'num_candidate_pairs': 4,
    'classifications': [{'drug': 'activated charcoal',
      'gene': 'EGFR',
      'interaction': False,
      'evidence': None,
      'interaction_type': None,
      'directionality': None,
      'error_message': 'The abstract discusses 4-anilinoquinazoline compounds as EGFR inhibitors. Activated charcoal is not mentioned in the text and has no supported interaction with EGFR based on this context.'},
     {'drug': 'activated charcoal',
      'gene': 'EGFR',
      'interaction': False,
      'evidence': None,
      'interaction_type': None,
      'directionality': None,
      'error_message': 'The abstract discusses 4-anilinoquinazoline compounds as EGFR inhibitors. Activated charcoal is not mentioned in the text and has

### Save

Save the output from a wags-llm run. After running through this whole process but I think its good.

In [13]:
rows = []

for run in stored_runs:
    for result in run["results"]:
        candidate_drugs = result.get("candidate_drugs", [])
        candidate_genes = result.get("candidate_genes", [])
        classifications = result.get("classifications", [])

        # preserve rows for skipped blocks / total failures
        if not classifications:
            rows.append(
                {
                    "run_idx": run["run_idx"],
                    "prompt_version": run["prompt_version"],
                    "temperature": run["temperature"],
                    "pmid": result.get("pmid"),
                    "block_id": result.get("block_id"),
                    "skipped": result.get("skipped", False),
                    "skip_reason": result.get("skip_reason"),

                    "candidate_drugs": ";".join(candidate_drugs),
                    "candidate_genes": ";".join(candidate_genes),
                    "candidate_drug_count": len(candidate_drugs),
                    "candidate_gene_count": len(candidate_genes),
                    "num_candidate_pairs": result.get("num_candidate_pairs", 0),

                    "drug": None,
                    "gene": None,
                    "interaction": None,
                    "interaction_type": None,
                    "directionality": None,
                    "evidence": None,

                    "error_message": result.get("error_message"),
                }
            )
            continue

        for classification in classifications:
            rows.append(
                {
                    "run_idx": run["run_idx"],
                    "prompt_version": run["prompt_version"],
                    "temperature": run["temperature"],
                    "pmid": result.get("pmid"),
                    "block_id": result.get("block_id"),
                    "skipped": result.get("skipped", False),
                    "skip_reason": result.get("skip_reason"),

                    "candidate_drugs": ";".join(candidate_drugs),
                    "candidate_genes": ";".join(candidate_genes),
                    "candidate_drug_count": len(candidate_drugs),
                    "candidate_gene_count": len(candidate_genes),
                    "num_candidate_pairs": result.get("num_candidate_pairs"),

                    "drug": classification.get("drug"),
                    "gene": classification.get("gene"),
                    "interaction": classification.get("interaction"),
                    "interaction_type": classification.get("interaction_type"),
                    "directionality": classification.get("directionality"),
                    "evidence": classification.get("evidence"),

                    "error_message": classification.get("error_message")
                    or result.get("error_message"),
                }
            )

df_results = pl.DataFrame(rows)

df_results.write_csv("2026-06-15-interaction_classifications-egfr.csv")
df_results.write_parquet("2026-06-15-interaction_classifications-egfr.parquet")

print(df_results.shape)
df_results.head()

(125, 19)


run_idx,prompt_version,temperature,pmid,block_id,skipped,skip_reason,candidate_drugs,candidate_genes,candidate_drug_count,candidate_gene_count,num_candidate_pairs,drug,gene,interaction,interaction_type,directionality,evidence,error_message
i64,str,i64,str,str,bool,str,str,str,i64,i64,i64,str,str,bool,str,str,str,str
0,"""v2""",0,"""19969465""","""0""",false,null,"""activated charcoal;tyrosine""","""EGF;EGFR""",2,2,4,"""activated charcoal""","""EGFR""",false,null,null,null,"""The abstract discusses 4-anili…"
0,"""v2""",0,"""19969465""","""0""",false,null,"""activated charcoal;tyrosine""","""EGF;EGFR""",2,2,4,"""activated charcoal""","""EGFR""",false,null,null,null,"""The abstract discusses 4-anili…"
0,"""v2""",0,"""19969465""","""0""",false,null,"""activated charcoal;tyrosine""","""EGF;EGFR""",2,2,4,"""tyrosine""","""EGFR""",false,null,null,null,"""The abstract discusses inhibit…"
0,"""v2""",0,"""19969465""","""0""",false,null,"""activated charcoal;tyrosine""","""EGF;EGFR""",2,2,4,"""tyrosine""","""EGFR""",false,null,null,null,"""The abstract discusses inhibit…"
0,"""v2""",0,"""24565969""","""1""",true,"""No normalized drug and gene ca…","""""","""EGFR""",0,1,0,null,null,null,null,null,null,null


In [ ]:
df_results['interaction'].value_counts()